# Clearance / Buffer Space Demo

This notebook shows how `buffer_space` is incorporated into the packing environment.

The important idea is that each item keeps its true physical dimension, but the environment can also use a larger virtual occupied dimension for EMS feasibility and overlap checks:

```text
true dimension    = (dx, dy, dz)
virtual dimension = (dx + buffer_space, dy + buffer_space, dz)
```

The true item is still used for height-map and feasibility map updates. The virtual item, on the other hand, reserves extra x/y clearance around the placement to update the EMS list.

In [10]:
from pathlib import Path
from copy import deepcopy
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "packing_env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import IFrame, display

from packing_env import PackingEnv
from packing_env.visualization import PackVisualizer

## Build Two Comparable Environments

Both environments use the same seed and item distribution. The only intended difference is `item_buffer_space`.

In [11]:
SEED = 0
CONTAINER_SIZE = (600, 600, 600)
BUFFER_CAPACITY = 12
K_PLACEMENT = 80
DS_NAME = "random"


def make_env(buffer_space: int) -> PackingEnv:
    env = PackingEnv(
        k_placement=K_PLACEMENT,
        ds_name=DS_NAME,
        buffer_capacity=BUFFER_CAPACITY,
        container_size=CONTAINER_SIZE,
        item_buffer_space=buffer_space,
        remove_inscribed_ems=True,
    )
    env.reset(seed=SEED)
    return env


env_no_clearance = make_env(buffer_space=0)
env_with_clearance = make_env(buffer_space=30)

print("Initial item without clearance:", env_no_clearance.selected_item.raw())
print("Initial item with clearance:   ", env_with_clearance.selected_item.raw())
print("Same sampled item:", np.array_equal(env_no_clearance.selected_item.raw(), env_with_clearance.selected_item.raw()))

Initial item without clearance: [240 120 300]
Initial item with clearance:    [240 120 300]
Same sampled item: True


## Inspect Candidate Counts

Clearance makes the virtual item larger in x/y, so fewer EMS candidates may be feasible.

In [12]:
def candidate_summary(env: PackingEnv) -> dict:
    obs = env.current_obs
    action_mask = obs["action_mask"].astype(bool)
    return {
        "buffer_space": env.item_buffer_space,
        "current_item": tuple(map(int, env.selected_item.raw())),
        "feasible_actions": int(action_mask.sum()),
        "feasible_no_rotation": int(action_mask[0].sum()),
        "feasible_rotated": int(action_mask[1].sum()),
    }


pd.DataFrame([
    candidate_summary(env_no_clearance),
    candidate_summary(env_with_clearance),
])

,buffer_space,current_item,feasible_actions,feasible_no_rotation,feasible_rotated
0,0,"(240, 120, 300)",2,1,1
1,30,"(240, 120, 300)",2,1,1


## Run A Small Policy Packing Loop

This is intentionally simple: at each step, choose the first feasible action from the mask. The goal is not to produce the best packing result, only to show how clearance propagates through placed items.

In [13]:
def first_feasible_action(obs) -> int | None:
    feasible = np.flatnonzero(obs["action_mask"].reshape(-1))
    if len(feasible) == 0:
        return None
    return int(feasible[0])


def record_pack_step(box, env) -> tuple:
    return (
        (int(box.FLB.x), int(box.FLB.y), int(box.FLB.z)),
        (int(box.Dim.dx), int(box.Dim.dy), int(box.Dim.dz)),
        bool(getattr(box, "rot", False)),
        env.buffer.dims(),
    )


def run_first_feasible(env: PackingEnv, max_steps: int = 8) -> tuple[pd.DataFrame, list[tuple]]:
    rows = []
    pack_history = []
    obs = env.current_obs
    for step in range(1, max_steps + 1):
        action = first_feasible_action(obs)
        if action is None:
            rows.append({
                "step": step,
                "buffer_space": env.item_buffer_space,
                "status": "blocked",
            })
            break

        obs, reward, done, truncated, info = env.step(action)
        placed = env.container.placed_items[-1]
        pack_history.append(record_pack_step(placed, env))
        rows.append({
            "step": step,
            "buffer_space": env.item_buffer_space,
            "status": "placed",
            "true_dim": tuple(map(int, placed.Dim.raw())),
            "virtual_dim": tuple(map(int, placed.Virtual_Dim.raw())),
            "position": (int(placed.FLB.x), int(placed.FLB.y), int(placed.FLB.z)),
            "reward_volume_ratio": float(reward),
            "utilization": float(env.container.utilization),
        })
        if done or truncated:
            break
    return pd.DataFrame(rows), pack_history


env_no_clearance = make_env(buffer_space=0)
env_with_clearance = make_env(buffer_space=30)

initial_buffer_no_clearance = env_no_clearance.buffer.dims()
initial_buffer_with_clearance = env_with_clearance.buffer.dims()

summary_no_clearance, pack_history_no_clearance = run_first_feasible(env_no_clearance, max_steps=8)
summary_with_clearance, pack_history_with_clearance = run_first_feasible(env_with_clearance, max_steps=8)

summary = pd.concat(
    [summary_no_clearance, summary_with_clearance],
    ignore_index=True,
)
summary


,step,buffer_space,status,true_dim,virtual_dim,position,reward_volume_ratio,utilization
0,1,0,placed,"(240, 120, 300)","(240, 120, 300)","(0, 0, 0)",0.040,0.040
1,2,0,placed,"(240, 300, 180)","(240, 300, 180)","(240, 0, 0)",0.060,0.100
2,3,0,placed,"(240, 180, 240)","(240, 180, 240)","(0, 120, 0)",0.048,0.148
3,4,0,placed,"(240, 120, 240)","(240, 120, 240)","(0, 300, 0)",0.032,0.180
4,5,0,placed,"(180, 240, 300)","(180, 240, 300)","(240, 300, 0)",0.060,0.240
5,6,0,placed,"(240, 240, 180)","(240, 240, 180)","(240, 0, 180)",0.048,0.288
6,7,0,placed,"(180, 300, 120)","(180, 300, 120)","(420, 300, 0)",0.030,0.318
7,8,0,placed,"(240, 300, 180)","(240, 300, 180)","(0, 120, 240)",0.060,0.378
8,1,30,placed,"(240, 120, 300)","(270, 150, 300)","(0, 0, 0)",0.040,0.040
9,2,30,placed,"(240, 300, 180)","(270, 330, 180)","(270, 0, 0)",0.060,0.100


Notice that `true_dim` is the same physical item dimension used for utilization, while `virtual_dim` grows by `buffer_space` in x/y when clearance is enabled.

In [14]:
comparison = pd.DataFrame([
    {
        "case": "no clearance",
        "buffer_space": env_no_clearance.item_buffer_space,
        "placed_items": len(env_no_clearance.container.placed_items),
        "utilization": env_no_clearance.container.utilization,
        "remaining_buffer_items": len(env_no_clearance.buffer.items),
    },
    {
        "case": "30 mm clearance",
        "buffer_space": env_with_clearance.item_buffer_space,
        "placed_items": len(env_with_clearance.container.placed_items),
        "utilization": env_with_clearance.container.utilization,
        "remaining_buffer_items": len(env_with_clearance.buffer.items),
    },
])
comparison

,case,buffer_space,placed_items,utilization,remaining_buffer_items
0,no clearance,0,8,0.378,12
1,30 mm clearance,30,8,0.378,12


## Visualize The Result With Interactive Replays

The replay is more informative than a final-state image because you can step forward/backward and rotate each frame. Each frame rebuilds the environment from the same initial buffer and applies the recorded placements one by one.


In [15]:
from packing_env.data_type.geometry import Orthogonal3D, Point3D
from packing_env.data_type.item import Item
from packing.interactive_replay import InteractiveReplayRecorder

REPLAY_DIR = PROJECT_ROOT / "outputs" / "plotly_live" / "clearance_demo"
REPLAY_DIR.mkdir(parents=True, exist_ok=True)


def save_clearance_replay(
    *,
    buffer_space: int,
    initial_buffer: list[tuple[int, int, int]],
    pack_history: list[tuple],
    title: str,
    out_path: Path,
) -> Path:
    replay_env = make_env(buffer_space=buffer_space)
    replay_env.buffer.buffer = [Orthogonal3D(*dims) for dims in initial_buffer]

    recorder = InteractiveReplayRecorder(str(out_path), interval_ms=700)

    def capture(frame_title: str) -> None:
        fig, _ = PackVisualizer(
            replay_env,
            title=frame_title,
            show_anchor=True,
            show_ems=False,
        ).refresh()
        recorder.capture(frame_title, fig)

    capture(f"{title}: Initial state")
    for step, (box_pos, box_dims, box_rot, buffer_after) in enumerate(pack_history, start=1):
        box = Item(
            FLB=Point3D(*box_pos),
            Dim=Orthogonal3D(*box_dims),
            buffer_space=buffer_space,
        )
        box.rot = box_rot
        replay_env.pack(box)
        replay_env.buffer.buffer = [Orthogonal3D(*dims) for dims in buffer_after]
        capture(f"{title}: Pack step {step}")

    recorder.save()
    return out_path


In [16]:
replay_no_clearance = save_clearance_replay(
    buffer_space=0,
    initial_buffer=initial_buffer_no_clearance,
    pack_history=pack_history_no_clearance,
    title="No clearance",
    out_path=REPLAY_DIR / "no_clearance_replay.html",
)

replay_with_clearance = save_clearance_replay(
    buffer_space=30,
    initial_buffer=initial_buffer_with_clearance,
    pack_history=pack_history_with_clearance,
    title="30 mm clearance",
    out_path=REPLAY_DIR / "clearance_30mm_replay.html",
)

print("No-clearance replay:", replay_no_clearance)
print("30 mm clearance replay:", replay_with_clearance)


Saved interactive replay: /home/gao/CJ_block_dockerize/outputs/plotly_live/clearance_demo/no_clearance_replay.html (9 frames)
Saved interactive replay: /home/gao/CJ_block_dockerize/outputs/plotly_live/clearance_demo/clearance_30mm_replay.html (9 frames)
No-clearance replay: /home/gao/CJ_block_dockerize/outputs/plotly_live/clearance_demo/no_clearance_replay.html
30 mm clearance replay: /home/gao/CJ_block_dockerize/outputs/plotly_live/clearance_demo/clearance_30mm_replay.html


In [17]:
display(IFrame(src="../" + str(replay_no_clearance.relative_to(PROJECT_ROOT)), width="100%", height=780))


In [18]:
display(IFrame(src="../" + str(replay_with_clearance.relative_to(PROJECT_ROOT)), width="100%", height=780))


## Try Different Clearance Values

`buffer_space` must be non-negative and divisible by the height-map resolution, currently `10 mm`.

In [8]:
for buffer_space in [0, 10, 20, 30, 40]:
    env = make_env(buffer_space=buffer_space)
    rows, _ = run_first_feasible(env, max_steps=8)
    print(
        f"buffer_space={buffer_space:>2} mm | "
        f"placed={len(env.container.placed_items):>2} | "
        f"utilization={env.container.utilization:.3f} | "
        f"last_status={rows.iloc[-1]['status']}"
    )

buffer_space= 0 mm | placed= 8 | utilization=0.378 | last_status=placed
buffer_space=10 mm | placed= 7 | utilization=0.318 | last_status=placed
buffer_space=20 mm | placed= 7 | utilization=0.318 | last_status=placed
buffer_space=30 mm | placed= 8 | utilization=0.378 | last_status=placed
buffer_space=40 mm | placed= 6 | utilization=0.288 | last_status=placed
